<a href="https://colab.research.google.com/github/acapodanno/agent-ai/blob/main/chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
!pip install langchain langchain-community sentence-transformers tiktoken langchain-text-splitters

In [37]:
import tiktoken

enc = tiktoken.get_encoding("o200k_base")

def count_tokens(txt:str) -> int:
  return len(enc.encode(txt))


In [38]:
documento = """
L'intelligenza artificiale sta trasformando il modo in cui le aziende
operano. Dai chatbot al riconoscimento delle immagini, le applicazioni
sono ormai ovunque.

Il machine learning è una branca dell'IA che permette ai sistemi di
imparare dai dati senza essere esplicitamente programmati. Gli algoritmi
analizzano grandi quantità di dati per identificare pattern e fare
previsioni.

Il deep learning, a sua volta, è una sottobranca del machine learning
che utilizza reti neurali con molti strati. Questi modelli hanno
rivoluzionato il riconoscimento vocale, la visione artificiale e
l'elaborazione del linguaggio naturale.

I Large Language Model come GPT-4 e Claude sono addestrati su enormi
corpus di testo. Imparano a predire il token successivo in una sequenza,
e da questo processo emergono capacità sorprendenti di ragionamento
e generazione del linguaggio.

Il processo di addestramento richiede enormi quantità di dati e potenza
computazionale. Una volta addestrati, questi modelli possono essere
adattati a task specifici tramite fine-tuning o prompt engineering.

Le applicazioni pratiche includono assistenti virtuali, generazione di
codice, analisi di documenti, traduzione automatica e molto altro.
La sfida principale rimane quella di garantire che i modelli siano
affidabili, sicuri e privi di bias.
"""

print(f"Documento: {len(documento)} caratteri, {count_tokens(documento)} token")


Documento: 1316 caratteri, 316 token


### Fixed-size-chunking

In [39]:
def fix_size_chuncks(txt:str,chunk_size:int=100, overlap:int=100) -> list[str]:
  tokens = enc.encode(txt)
  chunks = []
  start = 0
  while start < len(tokens):
    end = start + chunk_size
    chunk_token = tokens[start:end]
    chunk = enc.decode(chunk_token)
    chunks.append(chunk)
    start = end - overlap
  return chunks

chunks_fixed = fix_size_chuncks(
    documento,
    chunk_size=100,
    overlap=20
)
print(f"Documento: {len(chunks_fixed)}")
print(f"Token per chunk: {count_tokens(chunks_fixed[0])}")
for i,c in enumerate(chunks_fixed[:3],1):
  print(f"{i+1}: {count_tokens(c)}")
  print(c.strip())

Documento: 4
Token per chunk: 100
2: 100
L'intelligenza artificiale sta trasformando il modo in cui le aziende
operano. Dai chatbot al riconoscimento delle immagini, le applicazioni
sono ormai ovunque.

Il machine learning è una branca dell'IA che permette ai sistemi di
imparare dai dati senza essere esplicitamente programmati. Gli algoritmi
analizzano grandi quantità di dati per identificare pattern e fare
previsioni.

Il deep learning, a sua volta, è una sottobr
3: 100
pattern e fare
previsioni.

Il deep learning, a sua volta, è una sottobranca del machine learning
che utilizza reti neurali con molti strati. Questi modelli hanno
rivoluzionato il riconoscimento vocale, la visione artificiale e
l'elaborazione del linguaggio naturale.

I Large Language Model come GPT-4 e Claude sono addestrati su enormi
corpus di testo. Imparano a predire il token
4: 100
addestrati su enormi
corpus di testo. Imparano a predire il token successivo in una sequenza,
e da questo processo emergono capacità s

### Sentece Splitting

In [40]:
import re

def sentence_chunks(txt:str, max_token: int=200) -> list[str]:
  paragraphs = [p.strip() for p in txt.split("\n\n") if p.strip()]
  chunks = []
  current = ""
  for p in paragraphs:
    sentences = re.split(r"(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?)\s", p)
    for s in sentences:
      candidate = (current + " " + s).strip()
      if count_tokens(current + s) <= max_token:
        current = candidate
      else:
        if current:
          chunks.append(current)
        current = s
  if current:
    chunks.append(current)
  return chunks


In [41]:
chunks_sentence = sentence_chunks(
    documento,
    max_token=100
)
print(f"Documento: {len(chunks_sentence)}")
print(f"Token per chunk: {count_tokens(chunks_sentence[0])}")
for i,c in enumerate(chunks_sentence[:3],1):
  print(f"{i+1}: {count_tokens(c)}")
  print(c.strip())

Documento: 4
Token per chunk: 87
2: 87
L'intelligenza artificiale sta trasformando il modo in cui le aziende
operano. Dai chatbot al riconoscimento delle immagini, le applicazioni
sono ormai ovunque. Il machine learning è una branca dell'IA che permette ai sistemi di
imparare dai dati senza essere esplicitamente programmati. Gli algoritmi
analizzano grandi quantità di dati per identificare pattern e fare
previsioni.
3: 84
Il deep learning, a sua volta, è una sottobranca del machine learning
che utilizza reti neurali con molti strati. Questi modelli hanno
rivoluzionato il riconoscimento vocale, la visione artificiale e
l'elaborazione del linguaggio naturale. I Large Language Model come GPT-4 e Claude sono addestrati su enormi
corpus di testo.
4: 83
Imparano a predire il token successivo in una sequenza,
e da questo processo emergono capacità sorprendenti di ragionamento
e generazione del linguaggio. Il processo di addestramento richiede enormi quantità di dati e potenza
computazionale. 

### Recursive Splitting

In [42]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    separators=["\n\n","\n", " "]
)

chunks_recursive = splitter.split_text(documento)
print(f"Documento: {len(chunks_recursive)}")
for i,c in enumerate(chunks_recursive,1):
  print(f"{i}: {count_tokens(c)}")
  print(c.strip())


Documento: 5
1: 87
L'intelligenza artificiale sta trasformando il modo in cui le aziende
operano. Dai chatbot al riconoscimento delle immagini, le applicazioni
sono ormai ovunque.

Il machine learning è una branca dell'IA che permette ai sistemi di
imparare dai dati senza essere esplicitamente programmati. Gli algoritmi
analizzano grandi quantità di dati per identificare pattern e fare
previsioni.
2: 61
Il deep learning, a sua volta, è una sottobranca del machine learning
che utilizza reti neurali con molti strati. Questi modelli hanno
rivoluzionato il riconoscimento vocale, la visione artificiale e
l'elaborazione del linguaggio naturale.
3: 59
I Large Language Model come GPT-4 e Claude sono addestrati su enormi
corpus di testo. Imparano a predire il token successivo in una sequenza,
e da questo processo emergono capacità sorprendenti di ragionamento
e generazione del linguaggio.
4: 47
Il processo di addestramento richiede enormi quantità di dati e potenza
computazionale. Una volta add

### Hierarchy Chunking

In [43]:
def hierarchy_chunk(txt:str,parent_size=512,child_size=128,overlap=50)-> list[str]:
  tokens = enc.encode(txt)
  result = []
  parent_start = 0
  while parent_start < len(tokens):
    parent_end = parent_start + parent_size
    parent_chunk_tokens = tokens [parent_start:parent_end]
    parent_chunk = enc.decode(parent_chunk_tokens)

    children = []
    child_start = parent_start
    while child_start < min(parent_end ,len(tokens)):
      child_end = min(child_start + child_size,parent_end)
      child_chunk_tokens = tokens[child_start:child_end]
      children.append(child_chunk_tokens)
      child_start += child_size - overlap
    result.append(
        {
            "parent": parent_chunk,
            "children": children
        }
    )
    parent_start += parent_size - overlap
  return result

hier_chunk = hierarchy_chunk(documento)